# **Gold — Analyse comptable**
Pipeline Olist : indicateurs financiers à partir de la zone silver.

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum as _sum, avg, count, round as _round,
    countDistinct, date_format, max as _max,
    collect_set, concat_ws, create_map, lit, coalesce
)
from itertools import chain

spark = SparkSession.builder \
    .appName("olist-gold-account") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

### **Chargement des tables silver**

In [ ]:
df_orders = spark.read.parquet("../data/silver/orders/")
df_customers = spark.read.parquet("../data/silver/customers/")
df_items = spark.read.parquet("../data/silver/order_items/")
df_payments = spark.read.parquet("../data/silver/payments/")
df_products = spark.read.parquet("../data/silver/products/")
df_sellers = spark.read.parquet("../data/silver/sellers/")
df_pcnt = spark.read.parquet("../data/silver/product_category_name_translation/")

### **Segmentation des commandes par statut**

Pour savoir se le processus est finalisé, il faut analyser le statut de la commande. À partir de là, on peut identifier sa position parmi les trois segments suivants : **revenu reconnu, en transit, ou exclu**.

En analysant le regroupement par statut, on constate que la grande majorité des commandes, soit environ 97 %, ont été livrées (96 478 sur 99 441 commandes). Les commandes annulées ou indisponibles représentent 1 234 transactions. Enfin, les autres commandes non finalisées (qu'elles soient en cours d'envoi, de traitement, de paiement, de création ou de facturation) totalisent 1 729 commandes (~1,7 %).

Partant de ce constat, il est intéressant de segmenter ces catégories pour comprendre le chiffre d'affaires réel et obtenir des indicateurs sur le taux de désistement ou sur d'éventuels problèmes survenant lors du traitement de la commande.

In [ ]:
df_orders.groupBy("order_status").count().show()

In [ ]:
# Trois segments selon le statut de la commande
df_orders_delivered = df_orders.filter(col("order_status") == "delivered")

df_orders_in_transit = df_orders.filter(
    col("order_status").isin(["created", "approved", "processing", "invoiced", "shipped"])
)

df_orders_excluded = df_orders.filter(
    col("order_status").isin(["canceled", "unavailable"])
)

## **Chiffre d'affaires par segment**

En considérant os segments cités ci-dessus, il est important de comprendre le chiffre d'affaires qui y est lié, en récupérant l'identifiant du client ainsi que l'heure de l'achat.

Hors frais de port, le chiffre d'affaires des produits livrés s'élève à un total de plus de 13,2 millions de reais (soit 3,4 millions d'euros à l'époque), ce qui représente plus de 97 % du CA global. Ce montant est nettement plus significatif que les sommes de l'ordre de 273 000 R$ (~69 500 €)*. 

Quant au chiffre d'affaires lié aux commandes non abouties, il est inférieur à 25 000 €, ce qui représente moins de 1 % du chiffre d'affaires total de l'ensemble des commandes.

<center>

|Segment du Revenu | CA produits (R$)| Pourcentage du total |
|:---|:---|:---|
| Reconnu | 13 221 498,11 | 97,3 % |
| En transit | 272 902,63 | 2 % |
| Exclu | 97 242, 96 | 0,7 % |



In [ ]:
df_gold_revenue_delivered = df_items.join(
    df_orders_delivered.select("order_id", "customer_id", "order_purchase_timestamp"),
    "order_id", "inner"
)

df_gold_revenue_in_transit = df_items.join(
    df_orders_in_transit.select("order_id", "customer_id", "order_purchase_timestamp"),
    "order_id", "inner"
)

df_gold_revenue_excluded = df_items.join(
    df_orders_excluded.select("order_id", "customer_id", "order_purchase_timestamp"),
    "order_id", "inner"
)

In [ ]:
# Fonction pour calculer le chiffre d'affaires d'un segment donné
def calculate_revenue(df, name):
    result = df.agg(
        _round(_sum("price"), 2).alias("ca_produits"),
        _round(_sum("freight_value"), 2).alias("total_frais_livraison"),
        _round(_sum("price") + _sum("freight_value"), 2).alias("ca_total")
    )
    print(f"=== {name} ===")
    result.show()
    return result

ca_delivered = calculate_revenue(df_gold_revenue_delivered, "Revenu reconnu (delivered)")
ca_in_transit = calculate_revenue(df_gold_revenue_in_transit, "Revenu en transit (payé, non livré)")
ca_excluded = calculate_revenue(df_gold_revenue_excluded, "Exclu (annulé/indisponible)")

## Chiffre d'affaires mensuel

Ici, on peut visualiser l'évolution du revenu reconnu dans le temps.

Les données s'étendent de septembre 2016 à août 2018, à l'exception du mois de novembre 2016 qui est manquant.

Le pic de chiffre d'affaires a eu lieu en novembre 2017, atteignant une valeur de 987 765,37 R$.

La moyenne des valeurs se situe à 574 847,74 R$, sachant qu'en octobre et décembre 2016, les montants étaient particulièrement bas.
Il est possible d'observer une augmentation, suivie d'une stabilisation du chiffre d'affaires.

In [ ]:
df_monthly_revenue = df_gold_revenue_delivered.withColumn(
    "month", date_format(col("order_purchase_timestamp"), "yyyy-MM")
).groupBy("month").agg(
    _round(_sum("price"), 2).alias("ca_produits"),
    _round(_sum("freight_value"), 2).alias("total_frais_livraison"),
    _round(_sum("price") + _sum("freight_value"), 2).alias("ca_total")
).orderBy("month")

df_monthly_revenue.show(30)
# df_monthly_revenue.select("ca_produits").orderBy("ca_produits", ascending=False).show(23)
df_monthly_revenue.select(_round(avg("ca_produits"),2)).show()

df_monthly_revenue.select(countDistinct("month")).collect()[0][0]

## Validation — paiements par commande

L'écrasante majorité des commandes (96 479 sur ~99 441, soit ~97 %) ne comporte qu'un seul paiement, ce qui correspond au cas standard. Une minorité de commandes présente toutefois plusieurs paiements, jusqu'à 29 dans le cas extrême observé ci-dessous.

L'inspection de ce cas montre qu'il s'agit de **multiples vouchers** appliqués à une même commande, et non d'un problème technique ou de tentatives de paiement échouées. Certaines valeurs sont même à 0 R$, probablement des vouchers promotionnels sans valeur monétaire réelle.

Cette observation confirme la nécessité d'agréger `order_payments` par `order_id` avant toute jointure avec `orders`, sous peine de dupliquer les lignes pour les commandes à paiements multiples.

In [ ]:
# Combien de paiements par commande ?
payments_per_order = df_payments.groupBy("order_id").agg(count("*").alias("nb_payments"))
payments_per_order.groupBy("nb_payments").count().orderBy("nb_payments").show()

In [ ]:
# Inspection d'un cas extrême (commande avec le plus de paiements)
example_order_id = df_payments.groupBy("order_id").count().orderBy(col("count").desc()).first()["order_id"]
df_payments.filter(col("order_id") == example_order_id).orderBy("payment_sequential").show(30)

## Répartition des paiements par type

Valeur totale, nombre de transactions et part de chaque type.

Le **credit_card** domine très largement (78,34 % de la valeur totale, R$ 12 542 084,19), cohérent avec les habitudes de paiement échelonné du e-commerce brésilien. Le **boleto** arrive en deuxième position (17,92 %), avec une valeur moyenne par transaction relativement élevée (R$ 145,03), suggérant un usage pour des achats plus importants payés en une seule fois.

Le **voucher** présente la valeur moyenne la plus faible (R$ 65,70), cohérent avec un usage fractionné de coupons promotionnels — confirmé par les 6 transactions à valeur nulle identifiées, qui tirent légèrement cette moyenne vers le bas sans affecter la fiabilité du total agrégé.

Les 3 transactions **not_defined** (valeur nulle) sont un résidu de qualité de données négligeable, sans impact sur les totaux.

In [ ]:
total_payments = df_payments.agg(_sum("payment_value")).collect()[0][0]

df_gold_payment_breakdown = df_payments.groupBy("payment_type").agg(
    _round(_sum("payment_value"), 2).alias("valeur_totale"),
    count("*").alias("nb_transactions"),
    _round(avg("payment_value"), 2).alias("valeur_moyenne")
).withColumn(
    "pourcentage_valeur",
    _round((col("valeur_totale") / total_payments) * 100, 2)
).orderBy(col("valeur_totale").desc())

df_gold_payment_breakdown.show()

In [ ]:
# Qualité des données : transactions voucher à valeur nulle
zero_value_vouchers = df_payments.filter(
    (col("payment_type") == "voucher") & (col("payment_value") == 0)
).count()
print(f"Transactions voucher avec valeur zéro : {zero_value_vouchers}")

## Parcellement moyen par type de paiement

L'hypothèse initiale est confirmée : seul le **credit_card** permet un paiement échelonné, avec une moyenne de 3,51 mensualités. Tous les autres types de paiement (**boleto, voucher, debit_card, not_defined**) affichent une moyenne exacte de 1, confirmant qu'ils ne proposent pas de facilité de paiement fractionné dans ce dataset.

Ce résultat renforce l'intérêt du **credit_card** pour l'entreprise : au-delà de sa part dominante en valeur (78,34 %), il est aussi le seul levier permettant au client d'étaler sa dépense, ce qui peut favoriser des paniers plus élevés.

In [ ]:
df_installments_by_type = df_payments.groupBy("payment_type").agg(
    _round(avg("payment_installments"), 2).alias("parcelles_moyennes"),
    count("*").alias("nb_transactions")
).orderBy(col("nb_transactions").desc())

df_installments_by_type.show()

## Panier moyen par commande

Le panier moyen global s'établit à **R$ 160,27**, calculé après agrégation des paiements au niveau commande (et non au niveau transaction), afin de ne pas fausser la moyenne avec les commandes à paiements fractionnés.

La répartition par combinaison de moyens de paiement montre que le **credit_card seul** concentre la grande majorité des commandes (73 407) avec le panier moyen le plus élevé (R$ 166,32). À l'inverse, le **voucher seul** présente le panier moyen le plus faible (R$ 105,23), cohérent avec son usage pour des montants réduits observé précédemment.

La combinaison **credit_card + voucher** (2 210 commandes) se situe entre les deux (R$ 148,90), suggérant que le voucher est souvent utilisé en complément d'un paiement principal plutôt que comme mode de paiement unique pour de gros achats. Le cas `credit_card + debit_card` reste marginal (1 seule commande).

In [ ]:
df_payments_agg = df_payments.groupBy("order_id").agg(
    _sum("payment_value").alias("valeur_totale_payee"),
    _max("payment_installments").alias("max_parcelles"),
    concat_ws(",", collect_set("payment_type")).alias("types_paiement")
)

In [ ]:
df_orders_for_payment_analysis = df_orders.filter(
    ~col("order_status").isin(["canceled", "unavailable"])
)

df_gold_payments = df_payments_agg.join(
    df_orders_for_payment_analysis.select("order_id", "customer_id", "order_status", "order_purchase_timestamp"),
    "order_id", "inner"
)

In [ ]:
df_gold_payments.agg(
    _round(avg("valeur_totale_payee"), 2).alias("panier_moyen")
).show()

df_gold_payments.groupBy("types_paiement").agg(
    count("*").alias("nb_commandes"),
    _round(avg("valeur_totale_payee"), 2).alias("panier_moyen")
).orderBy(col("nb_commandes").desc()).show()

## Paiements par état du client (demande géographique)

**São Paulo (SP)** domine très largement, avec 37,47 % de la valeur totale payée et 41 745 commandes — plus du triple du deuxième état (RJ, 13,39 %). Avec le Minas Gerais (MG, 11,7 %), ces trois états du Sudeste concentrent à eux seuls plus de 62 % du chiffre d'affaires, cohérent avec la concentration démographique et économique historique de cette région au Brésil.

Un constat intéressant se dégage du panier moyen : les états les plus représentés en volume (SP, RJ, MG) n'ont pas les paniers moyens les plus élevés. Au contraire, des états moins peuplés comme **PB** (R$ 264,08), **AC** (R$ 242,97) ou **RO** (R$ 240,58) affichent les paniers moyens les plus hauts — possiblement lié à des coûts de livraison plus élevés vers ces régions plus isolées, intégrés dans le montant payé.

Les états du Nord (AM, AC, AP, RR) restent marginaux en valeur absolue (moins de 0,2 % chacun), reflet à la fois de la faible densité de population de ces états et d'une présence e-commerce encore limitée dans la région — probablement en raison de leur éloignement par rapport aux principaux centres de production et d'expédition, entraînant des frais de port plus élevés ainsi que des délais de livraison plus longs.

In [ ]:
total_customer_payments = df_payments_agg.join(
    df_orders.select("order_id", "customer_id"), "order_id", "inner"
).agg(_sum("valeur_totale_payee")).collect()[0][0]

df_payments_by_customer_state = df_payments_agg.join(
    df_orders.select("order_id", "customer_id"), "order_id", "inner"
).join(
    df_customers.select("customer_id", "customer_state"), "customer_id", "inner"
).groupBy("customer_state").agg(
    _round(_sum("valeur_totale_payee"), 2).alias("valeur_totale"),
    count("*").alias("nb_commandes"),
    _round(avg("valeur_totale_payee"), 2).alias("panier_moyen")
).withColumn(
    "pourcentage_valeur",
    _round((col("valeur_totale") / total_customer_payments) * 100, 2)
).orderBy(col("valeur_totale").desc())

df_payments_by_customer_state.show(27)

## Paiements par région du client

La région **Sudeste** confirme sa position dominante observée au niveau des états, concentrant R$ 10 340 831,46 (la grande majorité du chiffre d'affaires total) et 68 265 commandes — soit plus de quatre fois la région suivante. Le **Sul** et le **Nordeste** se partagent une part secondaire mais non négligeable, tandis que le **Centro-Oeste** et surtout le **Norte** restent marginaux, ce dernier confirmant à l'échelle régionale la faible présence e-commerce déjà observée pour ses états individuels (AM, AC, AP, RR).

Cette concentration géographique extrême sur le Sudeste constitue un facteur important à considérer pour toute stratégie logistique ou commerciale basée sur ces données.

In [ ]:
region_map = {
    "AC": "Norte", "AP": "Norte", "AM": "Norte", "PA": "Norte", "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste", "PB": "Nordeste",
    "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul"
}

mapping_expr = create_map([lit(x) for x in chain(*region_map.items())])

df_payments_by_customer_region = df_payments_by_customer_state.withColumn(
    "region", mapping_expr[col("customer_state")]
).groupBy("region").agg(
    _round(_sum("valeur_totale"), 2).alias("valeur_totale"),
    _sum("nb_commandes").alias("nb_commandes")
).orderBy(col("valeur_totale").desc())

df_payments_by_customer_region.show()

## Validation — vendeurs par commande

La quasi-totalité des commandes (97 388 sur ~99 441, soit ~98 %) implique un seul vendeur, validant l'hypothèse retenue pour l'analyse par état du vendeur et le top vendeurs : l'utilisation de `order_items.price` reste fiable, sans risque significatif de double comptage. Une minorité de commandes (environ 2 %) impliquent plusieurs vendeurs (jusqu'à 5 dans les cas extrêmes), ce qui ne remet pas en cause l'approche globale mais confirme qu'un rattachement direct du `payment_value` à un seul vendeur aurait été incorrect pour ces commandes.

In [ ]:
sellers_per_order = df_items.groupBy("order_id").agg(countDistinct("seller_id").alias("nb_sellers"))
sellers_per_order.groupBy("nb_sellers").count().orderBy("nb_sellers").show()

## Chiffre d'affaires par état du vendeur (offre géographique)

L'offre est encore plus concentrée que la demande : **São Paulo** génère seul 64,4 % du chiffre d'affaires (R$ 8 753 396,21), soit près de sept fois plus que le deuxième état (PR, 9,28 %). Cette concentration dépasse largement celle observée côté client (SP représentait 37,47 % de la demande), révélant un déséquilibre structurel entre l'origine de l'offre et celle de la demande — la majorité des produits expédiés depuis São Paulo doivent ainsi parcourir de longues distances pour atteindre les clients des autres régions, en particulier ceux du Nord et du Nordeste.

À l'inverse, des états comme la **Bahia** (BA) ou le **Paraíba** (PB) affichent un prix moyen par article très élevé (R$ 444,11 et R$ 449,87) malgré un volume marginal — suggérant la présence de quelques vendeurs spécialisés dans des produits à forte valeur unitaire plutôt qu'un volume de vente important. Plusieurs états (PA, AM, AC) ne comptent que quelques vendeurs isolés, confirmant que l'offre du Nord du pays est quasiment inexistante sur la plateforme.

In [ ]:
total_seller_revenue = df_items.agg(_sum("price")).collect()[0][0]

df_revenue_by_seller_state = df_items.join(
    df_sellers.select("seller_id", "seller_state"), "seller_id", "inner"
).groupBy("seller_state").agg(
    _round(_sum("price"), 2).alias("valeur_totale"),
    count("*").alias("nb_items"),
    _round(avg("price"), 2).alias("prix_moyen")
).withColumn(
    "pourcentage_valeur",
    _round((col("valeur_totale") / total_seller_revenue) * 100, 2)
).orderBy(col("valeur_totale").desc())

df_revenue_by_seller_state.show(30)

## Top vendeurs par chiffre d'affaires généré

Le top 20 confirme la domination de São Paulo observée à l'échelle de l'état : la grande majorité des meilleurs vendeurs y sont basés, avec des villes variées (guariba, ibitinga, sumare, itaquaquecetuba...), suggérant une diversification géographique de l'offre à l'intérieur même de l'état plutôt qu'une concentration sur la seule capitale.

Le classement révèle deux profils de vendeurs performants bien distincts. D'un côté, des vendeurs à **fort volume et prix unitaire modéré** (ex. : ibitinga, 1 949 articles à R$ 101,02 en moyenne ; sao jose do rio preto, 1 926 articles à R$ 55,38). De l'autre, des vendeurs à **faible volume mais prix unitaire élevé** (ex. : lauro de freitas/BA, seulement 400 articles mais à R$ 544,85 chacun ; barueri, 322 articles à R$ 515,47 ; teresopolis/RJ, 174 articles à R$ 454,86), qui parviennent à générer un chiffre d'affaires comparable grâce à un positionnement sur des produits à forte valeur.

Ce constat suggère que la performance commerciale sur la plateforme ne dépend pas d'une stratégie unique : le volume et le positionnement premium constituent deux voies tout aussi valables pour figurer parmi les meilleurs vendeurs.

In [ ]:
df_top_sellers = df_items.join(
    df_orders_delivered.select("order_id"), "order_id", "inner"
).join(
    df_sellers.select("seller_id", "seller_city", "seller_state"), "seller_id", "inner"
).groupBy("seller_id", "seller_city", "seller_state").agg(
    _round(_sum("price"), 2).alias("ca_genere"),
    count("*").alias("nb_items_vendus"),
    _round(avg("price"), 2).alias("prix_moyen")
).orderBy(col("ca_genere").desc())

df_top_sellers.show(20, truncate=False)

## Chiffre d'affaires par catégorie de produit

`product_category_name` déjà nettoyé en silver (valeur 'unknown' si absente).

Aucune catégorie ne domine de façon écrasante : **health_beauty** arrive en tête avec seulement 9,26 % du chiffre d'affaires, suivie de près par **watches_gifts** (8,87 %) et **bed_bath_table** (7,63 %). Cette répartition relativement équilibrée sur le top 30 contraste fortement avec la concentration géographique observée précédemment (Sudeste, São Paulo), suggérant un catalogue de produits diversifié plutôt qu'une dépendance à quelques familles d'articles.

Le prix moyen révèle des profils de catégories très différents : **computers** se distingue avec un prix moyen élevé (R$ 1 098,34) malgré un faible volume (203 articles), typique d'une catégorie à forte valeur unitaire. À l'inverse, des catégories comme **electronics** (R$ 57,91) ou **telephony** (R$ 71,21) génèrent un chiffre d'affaires significatif grâce au volume plutôt qu'au prix unitaire.

La catégorie **unknown** ne représente que 1,32 % du chiffre d'affaires (1 603 articles), confirmant que le traitement des valeurs manquantes effectué en silver concerne une part marginale et maîtrisée du catalogue.

In [ ]:
total_category_revenue = df_items.agg(_sum("price")).collect()[0][0]

df_revenue_by_category = df_items.join(
    df_products.select("product_id", "product_category_name"), "product_id", "left"
).join(
    df_pcnt.select("product_category_name", "product_category_name_english"),
    "product_category_name", "left"
).withColumn(
    "categorie",
    coalesce(col("product_category_name_english"), col("product_category_name"))
).groupBy("categorie").agg(
    _round(_sum("price"), 2).alias("valeur_totale"),
    count("*").alias("nb_items"),
    _round(avg("price"), 2).alias("prix_moyen")
).withColumn(
    "pourcentage_valeur",
    _round((col("valeur_totale") / total_category_revenue) * 100, 2)
).orderBy(col("valeur_totale").desc())

df_revenue_by_category.show(30)

## Sauvegarde en zone gold

In [ ]:
df_gold_revenue_delivered.write.mode("overwrite").parquet("../data/gold/account/revenue_delivered/")
df_gold_revenue_in_transit.write.mode("overwrite").parquet("../data/gold/account/revenue_in_transit/")
df_gold_revenue_excluded.write.mode("overwrite").parquet("../data/gold/account/revenue_excluded/")
df_monthly_revenue.write.mode("overwrite").parquet("../data/gold/account/monthly_revenue/")
df_gold_payment_breakdown.write.mode("overwrite").parquet("../data/gold/account/payment_breakdown/")
df_installments_by_type.write.mode("overwrite").parquet("../data/gold/account/installments_by_type/")
df_payments_by_customer_state.write.mode("overwrite").parquet("../data/gold/account/payments_by_customer_state/")
df_payments_by_customer_region.write.mode("overwrite").parquet("../data/gold/account/payments_by_customer_region/")
df_revenue_by_seller_state.write.mode("overwrite").parquet("../data/gold/account/revenue_by_seller_state/")
df_top_sellers.write.mode("overwrite").parquet("../data/gold/account/top_sellers/")
df_revenue_by_category.write.mode("overwrite").parquet("../data/gold/account/revenue_by_category/")